# Accessing AWS KB


In [12]:
%pip install --upgrade "boto3>=1.35.50" "botocore>=1.35.50"
%pip install -U retrying
%pip install pprint

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
ERROR: Could not find a version that satisfies the requirement pprint (from versions: none)
ERROR: No matching distribution found for pprint
Note: you may need to restart the kernel to use updated packages.


In [4]:
import boto3
import pprint

In [ ]:
def new_session_with_token(access_key, secret_key, session_token, region="us-east-1"):
    return boto3.Session(
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        aws_session_token=session_token,
        region_name=region
    )

boto3_session = new_session_with_token(
    access_key="",
    secret_key",
    session_token=""
)

boto3_session.client("sts").get_caller_identity()

{'UserId': 'AROARQ2AAR4FC62QE5GJI:sgil@itglue.com',
 'Account': '104824082186',
 'Arn': 'arn:aws:sts::104824082186:assumed-role/AWSReservedSSO_GeneralAdministratorAccess_143e80dc68dc5f36/sgil@itglue.com',
 'ResponseMetadata': {'RequestId': 'c30eb968-591f-413e-961c-342593f7bd9e',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'c30eb968-591f-413e-961c-342593f7bd9e',
   'x-amz-sts-extended-request-id': 'MTp1cy1lYXN0LTE6MTc1NjEzNjE3MTQwNTpSOkcyd1I5WjdB',
   'content-type': 'text/xml',
   'content-length': '495',
   'date': 'Mon, 25 Aug 2025 15:36:11 GMT'},
  'RetryAttempts': 0}}

In [8]:
# try out KB using RetrieveAndGenerate API
region_name = boto3_session.region_name
bedrock_agent_runtime_client = boto3_session.client("bedrock-agent-runtime", region_name=region_name)
model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0" 
model_arn = f'arn:aws:bedrock:us-east-1:104824082186:inference-profile/{model_id}'

In [10]:
# set values for the knowledgebase
kb_id = "R2P64GU0E2"

In [ ]:
query = """
support team is saying that they remember that they used to inform users that they require Microsoft P1 or P2 developer license to be able to rotate passwords for multitenant.
Is it still mandatory to have p1 or p2 or will it work if they have direct access, and does direct access means they have access to the tenant where they can login to admin centre using global admin creds on tenant?
"""
response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        'text': query
    },
    retrieveAndGenerateConfiguration={
        'type': 'KNOWLEDGE_BASE',
        'knowledgeBaseConfiguration': {
            'knowledgeBaseId': kb_id,
            'modelArn': model_arn
        }
    },
)

generated_text = response['output']['text']

In [14]:
print(generated_text)

According to the information provided, if you have direct access to multi-tenants, you do not need any P1 or P2 Microsoft licenses. This is explicitly stated in the documentation.

Direct access appears to refer to having administrative access to the tenant where you can log in to the admin center. The documentation mentions steps for accessing tenants through the Microsoft admin center and switching directories, which suggests that direct access involves having global admin credentials for the tenant. For tenants without direct access, the documentation indicates that you would need at least one of these licenses:
- Microsoft 365 E5 Developer license
- Microsoft 365 E5 Developer (without Windows and Audio Conferencing)
- Microsoft Entra ID Governance
- Microsoft Entra ID P2

The process involves checking for these licenses, purchasing one if not available, and assigning it to a user in the tenant.
